# Credit Card Fraud Detection with Graph Neural Networks

This project uses a **Graph Neural Network (GNN)** built with **PyTorch Geometric** to detect fraudulent credit-card transactions. It compares the GNN with a normal neural network (MLP) that sees each transaction on its own.

**Data:** IBM synthetic credit-card transactions (Kaggle), one customer (User 0): 19,963 transactions, 27 of them fraudulent.

**Idea:** a fraudulent transaction often only looks suspicious *in context*, for example a sudden online purchase on a card that is usually swiped. A GNN lets each transaction look at its neighbours: the card's other recent transactions and the merchant's other recent transactions.

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.color': '#e6e6e6',
                     'axes.axisbelow': True})
BLUE, ORANGE, GREEN = '#2a78d6', '#eb6834', '#1baf7a'

## 2. Load and clean the data

In [2]:
df = pd.read_csv('User0_credit_card_transactions.csv')
print(df.shape)
df.head()

(19963, 15)


,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,No
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,No
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,No


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19963 entries, 0 to 19962
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   User            19963 non-null  int64  
 1   Card            19963 non-null  int64  
 2   Year            19963 non-null  int64  
 3   Month           19963 non-null  int64  
 4   Day             19963 non-null  int64  
 5   Time            19963 non-null  str    
 6   Amount          19963 non-null  str    
 7   Use Chip        19963 non-null  str    
 8   Merchant Name   19963 non-null  int64  
 9   Merchant City   19963 non-null  str    
 10  Merchant State  18646 non-null  str    
 11  Zip             18316 non-null  float64
 12  MCC             19963 non-null  int64  
 13  Errors?         574 non-null    str    
 14  Is Fraud?       19963 non-null  str    
dtypes: float64(1), int64(7), str(7)
memory usage: 2.3 MB


In [4]:
df.isnull().sum()

User                  0
Card                  0
Year                  0
Month                 0
Day                   0
Time                  0
Amount                0
Use Chip              0
Merchant Name         0
Merchant City         0
Merchant State     1317
Zip                1647
MCC                   0
Errors?           19389
Is Fraud?             0
dtype: int64

* `Merchant State` and `Zip` are missing for online transactions, which have no physical location. They aren't used as features.
* `Errors?` is missing when there was no error, so it is filled with "None".

Other cleaning steps: remove the `$` from `Amount`, combine the date and time columns into one timestamp, convert the label to 1 / 0, and sort by time.

In [5]:
df['Amount'] = df['Amount'].str.replace('$', '', regex=False).astype(float)
df['Datetime'] = pd.to_datetime(df[['Year', 'Month', 'Day']]) + pd.to_timedelta(df['Time'] + ':00')
df['Errors?'] = df['Errors?'].fillna('None')
df['Is Fraud?'] = (df['Is Fraud?'] == 'Yes').astype(int)
df['Use Chip'] = df['Use Chip'].str.replace(' Transaction', '')
df = df.sort_values('Datetime').reset_index(drop=True)

df[['Card', 'Datetime', 'Amount', 'Use Chip', 'Merchant Name', 'Merchant City', 'MCC', 'Errors?', 'Is Fraud?']].head()

,Card,Datetime,Amount,Use Chip,Merchant Name,Merchant City,MCC,Errors?,Is Fraud?
0,0,2002-09-01 06:21:00,134.09,Swipe,3527213246127876953,La Verne,5300,None,0
1,0,2002-09-01 06:42:00,38.48,Swipe,-727612092139916043,Monterey Park,5411,None,0
2,0,2002-09-02 06:22:00,120.34,Swipe,-727612092139916043,Monterey Park,5411,None,0
3,0,2002-09-02 17:45:00,128.95,Swipe,3414527459579106770,Monterey Park,5651,None,0
4,0,2002-09-03 06:23:00,104.71,Swipe,5817218446178736267,La Verne,5912,None,0
